In [5]:
import pandas as pd
import networkx as nx
from itertools import combinations

fname_base = 'DyHANE/2022/'

- a  MultiDiGraph with nodes of type A and edges from APA, APPA, AIA (weighted)
- a bipartite graph for A and each other type (AP, AI)

In [2]:
def create_mp_graph(df, source_col, target_col, weigth_col, key_col):
    G = nx.from_pandas_edgelist(df, source=source_col, target=target_col, edge_attr=weigth_col, edge_key=key_col, create_using=nx.DiGraph)
    return G

def create_mp_multi_graph(df, source_col, target_col, weigth_col, key_col):
    G = nx.from_pandas_edgelist(df, source=source_col, target=target_col, edge_attr=weigth_col, edge_key=key_col, create_using=nx.MultiDiGraph)
    return G

def create_mp_graph_undirect(df, source_col, target_col, weigth_col, key_col):
    G = nx.from_pandas_edgelist(df, source=source_col, target=target_col, edge_attr=weigth_col, edge_key=key_col, create_using=nx.Graph)
    return G 

from networkx.algorithms import bipartite
def create_bipartite_graph(df, A_col, other_col):
    G = nx.Graph()
    G.add_nodes_from(df[A_col], bipartite=0)
    G.add_nodes_from(df[other_col], bipartite=1)
    G.add_edges_from(df[[A_col, other_col]].values)
    return G


In [7]:
AA = pd.read_csv(fname_base+'AA_all.csv')

#Create bipartite graph AP
#AP = pd.read_csv(fname_base+'PA.csv')
AP = AA[AA['type'] == 'APA']
#G_AP = create_bipartite_graph(AP, 'author', 'paper')
G_AP = create_bipartite_graph(AP, 'source', 'target')

#Create bipartite graph AI
#AI = pd.read_csv(fname_base+'AI.csv')
AI = AA[AA['type'] == 'AIA']
#G_AI = create_bipartite_graph(AI, 'author_id', 'institution_id')
G_AI = create_bipartite_graph(AI, 'source', 'target')

In [8]:
#meta-path generation

 #compute all combinations (pairs)

def all_pairs(lista):
    return list(combinations(lista, 2))

def metapath_generation(fname_in, A_col, other_col, fname_out): #df has 2 columns in the format 'author_id', 'other_id'
    
    df_mp = pd.DataFrame(columns=['author1_id', 'author2_id'])

    #process chunks
    chunksize = 500
    for chunk in pd.read_csv(fname_in, chunksize=chunksize):

        '''
        column_names = chunk.columns
        target_name = column_names[1] #author_id
        other_name = column_names[0] #paper_id or institution_id #cambia
        '''

        #for each intermediate node, create list of authors
        df_temp = chunk.groupby(by=other_col, as_index=False).agg(list)

        #compute all combinations (pairs)
        df_temp[A_col] = df_temp[A_col].apply(all_pairs)
        df_temp = df_temp[df_temp[A_col].str.len() != 0]
        df_temp = df_temp.explode(A_col)

        #df_mp is now in the form : 'other_id' - pair of 'author_id'. Split 2 author ids
        df_new = pd.DataFrame()
        df_new[['author1_id','author2_id']] = df_temp[A_col].to_list()

        df_mp = pd.concat([df_mp, df_new], axis=0)
    
    #df_mp = df_mp.pivot_table(index = ['author_id1', 'author_id2'], aggfunc ='size', values='count')
    df_mp_ok = df_mp.groupby(['author1_id','author2_id']).size().reset_index(name='count')
    df_mp_ok.head(3)

    df_mp_ok.to_csv(fname_out, index=False)

In [5]:
#Create mp based graph

#prepare data
#authors --> APA, AIA, APPA

#APA
fname_in = fname_base+'PA.csv'
fname_out = fname_base+'APA.csv'
metapath_generation(fname_in, 'author', 'paper', fname_out)
#count: [1, 2, 3, 7, 6, 4, 8, 5, 29]

#AIA
fname_in = fname_base+'AI.csv'
fname_out = fname_base+'AIA.csv'
metapath_generation(fname_in, 'author_id', 'institution_id', fname_out)
#count: [1, 2, 4, 3, 6, 7, 21, 37, 36, 12, 28, 5]

#APPA
#richiede l'incrocio AP - PP - PA --> a1 a2 se esistono a1p1, a2p2 e p1p2
# vedi sotto

KeyboardInterrupt: 

In [25]:
# processing APPA 

df_PP = pd.read_csv(fname_base+"PP.csv") #citing - cited --> p1p2: p1 cites p2
df_PP_sub = df_PP[['citing', 'cited']]
df_PP_sub.rename(columns={'citing': 'paper'}, inplace=True)

df_PA = pd.read_csv(fname_base+"PA.csv") # paper - author
df_PA_sub = df_PA[['paper', 'author']]

df_APP = pd.merge(df_PA_sub, df_PP_sub, on='paper')
df_APP.rename(columns={'paper': 'paper1'}, inplace=True)
df_APP.rename(columns={'author': 'author1'}, inplace=True)
df_APP.rename(columns={'cited': 'paper'}, inplace=True)

df_APPA = pd.merge(df_APP, df_PA_sub, on='paper')
df_APPA.rename(columns={'paper': 'paper2'}, inplace=True)
df_APPA.rename(columns={'author': 'author2'}, inplace=True)

df_AA = df_APPA[['author1', 'author2']]

df_AA = df_AA.groupby(['author1', 'author2']).size().reset_index(name='count')
df_AA.to_csv(fname_base+'APPA.csv', index=False)

C:\Users\Liliana\AppData\Local\Temp\ipykernel_19292\3630872575.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_PP_sub.rename(columns={'citing': 'paper'}, inplace=True)


In [14]:
#check se AA_all diretto ok lo è

AA_all = pd.read_csv(fname_base+'AA_all.csv')
#AA_all.head(10)

sources = AA_all[AA_all['type'] == 'AIA']['source'].drop_duplicates().tolist()
targets = AA_all[AA_all['type'] == 'AIA']['target'].drop_duplicates().tolist()

print(len(sources))
print(len(targets))
A = [a for a in sources if a in targets]
print(len(A))


1685
1685
1685


In [27]:
# generate network of authors
#creation of unique dataframe: source - target - type - weight

df = pd.DataFrame(columns=['source', 'target', 'type', 'weight'])

#APA
APA = pd.read_csv(fname_base+'APA.csv')
APA.rename(columns={'author1_id': 'source', 'author2_id': 'target', 'count': 'weight'}, inplace=True)
#creare altra direzione
APA2 = pd.read_csv(fname_base+'APA.csv')
APA2.rename(columns={'author2_id': 'source', 'author1_id': 'target', 'count': 'weight'}, inplace=True)
APA = pd.concat([APA, APA2], axis=0)
#remove duplicates
APA = APA.groupby(['source', 'target']).sum().reset_index() #name='weight'
APA['type'] = 'APA'
#add to unique df
df = pd.concat([df, APA], axis=0)
print(df.shape)

#AIA
AIA = pd.read_csv(fname_base+'AIA.csv')
AIA.rename(columns={'author1_id': 'source', 'author2_id': 'target', 'count': 'weight'}, inplace=True)
#creare altra direzione
AIA2 = pd.read_csv(fname_base+'AIA.csv')
AIA2.rename(columns={'author2_id': 'source', 'author1_id': 'target', 'count': 'weight'}, inplace=True)
AIA = pd.concat([AIA, AIA2], axis=0)
#remove duplicates
AIA = AIA.groupby(['source', 'target']).sum().reset_index() #name='weight'
AIA['type'] = 'AIA'
#add to unique df
df = pd.concat([df, AIA], axis=0)
print(df.shape)

#APPA
APPA = pd.read_csv(fname_base+'APPA.csv')
APPA.rename(columns={'author1': 'source', 'author2': 'target', 'count': 'weight'}, inplace=True)
#creare altra direzione NO in questo caso
#remove duplicates
APPA = APPA.groupby(['source', 'target']).sum().reset_index() #name='weight'
APPA['type'] = 'APPA'
#add to unique df
df = pd.concat([df, APPA], axis=0)
print(df.shape)

df.to_csv(fname_base+'AA.csv', index=False)

(44579, 4)
(49532, 4)
(49834, 4)


In [15]:
#OK da qui se ho AA all

df = pd.read_csv(fname_base+'AA_all.csv') #AA.csv
G_A = create_mp_graph(df, 'source', 'target', 'weight', 'type')
G_A_multi = create_mp_multi_graph(df, 'source', 'target', 'weight', 'type')

In [32]:
#UNDIRECT GRAPH

# generate network of authors
#creation of unique dataframe: source - target - type - weight

df = pd.DataFrame(columns=['source', 'target', 'type', 'weight'])

#APA
APA = pd.read_csv(fname_base+'APA.csv')
APA.rename(columns={'author1_id': 'source', 'author2_id': 'target', 'count': 'weight'}, inplace=True)
#remove duplicates
APA = APA.groupby(['source', 'target']).sum().reset_index() #name='weight'
APA['type'] = 'APA'
#add to unique df
df = pd.concat([df, APA], axis=0)
print(df.shape)

#AIA
AIA = pd.read_csv(fname_base+'AIA.csv')
AIA.rename(columns={'author1_id': 'source', 'author2_id': 'target', 'count': 'weight'}, inplace=True)
#remove duplicates
AIA = AIA.groupby(['source', 'target']).sum().reset_index() #name='weight'
AIA['type'] = 'AIA'
#add to unique df
df = pd.concat([df, AIA], axis=0)
print(df.shape)

#APPA
APPA = pd.read_csv(fname_base+'APPA.csv')
APPA.rename(columns={'author1': 'source', 'author2': 'target', 'count': 'weight'}, inplace=True)
#remove duplicates
APPA = APPA.groupby(['source', 'target']).sum().reset_index() #name='weight'
APPA['type'] = 'APPA'
#add to unique df
df = pd.concat([df, APPA], axis=0)
print(df.shape)

df.to_csv(fname_base+'AA_undirect.csv', index=False)

(22411, 4)
(25008, 4)
(25310, 4)


In [5]:
df = pd.read_csv(fname_base+'AA_undirect.csv')
G_A_un = create_mp_graph_undirect(df, 'source', 'target', 'weight', 'type')

In [17]:
G_AP.nodes(data=True)

NodeDataView({'A5000004709': {'bipartite': 1}, 'A5000014798': {'bipartite': 1}, 'A5000015851': {'bipartite': 1}, 'A5000021578': {'bipartite': 1}, 'A5000053244': {'bipartite': 1}, 'A5000058426': {'bipartite': 1}, 'A5000062069': {'bipartite': 1}, 'A5000097985': {'bipartite': 1}, 'A5000100573': {'bipartite': 1}, 'A5000100609': {'bipartite': 1}, 'A5000104272': {'bipartite': 1}, 'A5000115067': {'bipartite': 1}, 'A5000128668': {'bipartite': 1}, 'A5000141065': {'bipartite': 1}, 'A5000147676': {'bipartite': 1}, 'A5000151104': {'bipartite': 1}, 'A5000151610': {'bipartite': 1}, 'A5000179526': {'bipartite': 1}, 'A5000193182': {'bipartite': 1}, 'A5000219510': {'bipartite': 1}, 'A5000223219': {'bipartite': 1}, 'A5000231809': {'bipartite': 1}, 'A5000232046': {'bipartite': 1}, 'A5000242314': {'bipartite': 1}, 'A5000247474': {'bipartite': 1}, 'A5000261188': {'bipartite': 1}, 'A5000266144': {'bipartite': 1}, 'A5000322730': {'bipartite': 1}, 'A5000324864': {'bipartite': 1}, 'A5000326339': {'bipartite': 

In [18]:
#num of nodes A
#pd.read_csv(fname_base+'A_labels.csv').shape[0] #13116
nA = G_A.number_of_nodes() #9173

A_fromAP = {n for n, d in G_AP.nodes(data=True) if d["bipartite"] == 1} #0
nA_P = len(A_fromAP) #13116
#P = set(G_AP) - nA_P

A_fromAI = {n for n, d in G_AI.nodes(data=True) if d["bipartite"] == 1} #0
nA_I = len(A_fromAI) #3317
#I = set(G_AI) - nA_I

print(nA, nA_P, nA_I) 

10769 8882 1685


In [19]:
#DIRECTED GRAPH WITH SELF-LOOPS (ALL NODES)

other_A = list(set(list(A_fromAP)+list(A_fromAI)))
print(len(other_A))

G_A_all = nx.DiGraph()
G_A_all.add_nodes_from(G_A)
G_A_all.add_nodes_from(other_A)

print(G_A_all.number_of_nodes())

9135
10769


In [21]:
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx

data = from_networkx(G_A_all)
print(data.edge_index)

tensor([], size=(2, 0), dtype=torch.int64)


    CENTRALITY MEASURES:
    Average degree (A: in_degree and out_degree for heterogeneous graph + MP based graphs)
    Degree assortativity (assesses the tendency of nodes to connect with similar nodes based on their degree) (A?)

In [8]:
#AVERAGE DEGREE (A)

numA = pd.read_csv(fname_base+'A_labels.csv').shape[0]

degs_AP = [deg for a, deg in G_AP.degree(A_fromAP)]
degs_AI = [deg for a, deg in G_AI.degree(A_fromAI)]
degs_AA = [deg for a, deg in G_A.in_degree()]
degs_AA_w = [deg for a, deg in G_A.in_degree(weight='weight')] 

def avg_in_degree(degs, numA):
    return float("{:.2f}".format(sum(degs)/numA))

print('Numbers of A:', numA)
print('Numbers of A connected with at least 1 P:', len(degs_AP))
print('Avg in-degree from P', avg_in_degree(degs_AP, numA))
print('Numbers of A connected with at least 1 I:', len(degs_AI))
print('Avg in-degree from I', avg_in_degree(degs_AI, numA))
print('Numbers of A connected with at least 1 A based on a meta-path of any type:', len(degs_AA))
print('Avg in-degree from A', avg_in_degree(degs_AA, numA))
print('Avg weighted in-degree from A', avg_in_degree(degs_AA_w, numA))


Numbers of A: 13116
Numbers of A connected with at least 1 P: 13116
Avg in-degree from P 1.22
Numbers of A connected with at least 1 I: 3317
Avg in-degree from I 0.26
Numbers of A connected with at least 1 A based on a meta-path of any type: 9173
Avg in-degree from A 3.61
Avg weighted in-degree from A 4.17


Degree assortativity of the unique (direct) meta-path based (sub)graph: 0.97


CONNECTIVITY MEASURES
Transitivity + clustering coeff. (A)
Density

In [10]:
#Transitivity #not implemented for multigraph type

trans = nx.transitivity(G_A)
print('Transitivity of the meta-path based (sub)graph:', "{:.2f}".format(trans))

#How to compute transitivity taking into account isolated nodes?
trans_all = nx.transitivity(G_A_all)
print('Transitivity of the meta-path based graph + isolated nodes:', "{:.2f}".format(trans_all))

Transitivity of the meta-path based (sub)graph: 0.96
Transitivity of the meta-path based graph + isolated nodes: 0.00


In [11]:
#clustering coefficient
cl = nx.average_clustering(G_A)
print('Average clustering coefficient of the meta-path based (sub)graph:', "{:.2f}".format(cl))

Average clustering coefficient of the meta-path based (sub)graph: 0.65


In [40]:
#Density

dens = nx.density(G_A)
print('Density of the meta-path based (sub)graph:', "{:.6f}".format(dens))

dens2 = nx.density(G_A_all)
print('Density of the meta-path based (sub)graph + isolated nodes:', "{:.6f}".format(dens2))

Density of the meta-path based (sub)graph: 0.000563
Density of the meta-path based (sub)graph + isolated nodes: 0.000000


PATH-BASED MEASURES
Average path length --> the average path length which is defined as the average of the shortest distances over all the pairs of nodes in the graph.
Diameter


In [23]:
#since the graph is disconnected, we compute the average path length and the diameter of the larger CC

largest_cc = max(nx.connected_components(G_A_un), key=len)
print('Number of nodes in the largest connected component:', len(largest_cc))
CC = G_A_un.subgraph(largest_cc).copy()
apl = nx.average_shortest_path_length(CC)
print('Average path length of the largest CC:', "{:.2f}".format(apl))

diam = nx.diameter(CC)
print('Diameter of the largest CC:', "{:.2f}".format(diam))

Number of nodes in the largest connected component: 897
Average path length of the largest CC: 13.01
Diameter of the largest CC: 33.00


MESOSCOPIC MEASURES
Strongly/weakly connected components
Communities?
Modularity

In [14]:
#Connected components

num_strong_cc = nx.number_strongly_connected_components(G_A)
num_weak_cc = nx.number_weakly_connected_components(G_A)

print('Number of strongly connected components:', num_strong_cc)
print('Number of weakly connected components:', num_weak_cc)

Number of strongly connected components: 2355
Number of weakly connected components: 2309


In [33]:
communities = nx.community.louvain_communities(G_A)
print('Number of communities', len(communities))

Number of communities 2769


In [35]:
#Communities

communities_un = nx.community.louvain_communities(G_A_un)
print('Number of communities', len(communities_un))

Number of communities 2327


In [36]:
#Modularity
mod = nx.community.modularity(G_A, communities)
print('Modularity:', "{:.2f}".format(mod))

Modularity: 0.93


In [37]:
#Modularity
mod_un = nx.community.modularity(G_A_un, communities_un)
print('Modularity:', "{:.2f}".format(mod_un))

Modularity: 0.97


ADD ATTRIBUTES FOR ATTRIBUTE ASSORTATIVITY

In [8]:
def attribute_dict(df, attr_name):
    df = df[['id', attr_name]]
    return df.set_index('id')[attr_name].to_dict()

In [4]:
df = pd.read_csv('D:\\PhD\\Openalex\\subset_ok\\years\\2019-2022\\DyHANE\\2019-2021\\features\\AX.csv')
authors = df['id'].tolist()
print(len(authors))

13116


In [6]:
#id - n_works - n_cit - impact_factor h_index i10_index
df_attr = pd.read_csv('D:\\PhD\\Openalex\\subset_ok\\years\\2019-2022\\attributes\\authors_numeric.csv')
print(df_attr.shape)
df_attr_sub = df_attr[df_attr['id'].isin(authors)]
print(df_attr_sub.shape)


(15397, 6)
(13116, 6)


In [17]:
df_lab = pd.read_csv(fname_base+'A_labels.csv')
df_lab.rename(columns={'author_id': 'id'}, inplace=True)
df_lab.shape

(13116, 2)

In [15]:
dict_labels = attribute_dict(df_lab, 'first_concept_name')
nx.set_node_attributes(G_A, dict_labels, name="label")

dict_n_works = attribute_dict(df_attr_sub, 'n_works')
nx.set_node_attributes(G_A, dict_n_works, name="n_works")

dict_n_cit = attribute_dict(df_attr_sub, 'n_cit')
nx.set_node_attributes(G_A, dict_n_cit, name="n_cit")

dict_impact_factor = attribute_dict(df_attr_sub, 'impact_factor')
nx.set_node_attributes(G_A, dict_impact_factor, name="impact_factor")

dict_h_index = attribute_dict(df_attr_sub, 'h_index')
nx.set_node_attributes(G_A, dict_h_index, name="h_index")

dict_i10_index = attribute_dict(df_attr_sub, 'i10_index')
nx.set_node_attributes(G_A, dict_i10_index, name="i10_index")


In [18]:
#attribute assortativity
lab_ass = nx.attribute_assortativity_coefficient(G_A, 'label')
print('Attribute assortativity for first_concept_name', "{:.2f}".format(lab_ass))

nw_ass = nx.attribute_assortativity_coefficient(G_A, 'n_works')
print('Attribute assortativity for n_works', "{:.2f}".format(nw_ass))

nc_ass = nx.attribute_assortativity_coefficient(G_A, 'n_cit')
print('Attribute assortativity for n_cit', "{:.2f}".format(nc_ass))

if_ass = nx.attribute_assortativity_coefficient(G_A, 'impact_factor')
print('Attribute assortativity for impact_factor', "{:.2f}".format(if_ass))

hi_ass = nx.attribute_assortativity_coefficient(G_A, 'h_index')
print('Attribute assortativity for h_index', "{:.2f}".format(hi_ass))

ii_ass = nx.attribute_assortativity_coefficient(G_A, 'i10_index')
print('Attribute assortativity for i10_index', "{:.2f}".format(ii_ass))

Attribute assortativity for first_concept_name 0.39
Attribute assortativity for n_works 0.09
Attribute assortativity for n_cit 0.15
Attribute assortativity for impact_factor 0.18
Attribute assortativity for h_index 0.16
Attribute assortativity for i10_index 0.18
